In [10]:
from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker
import ollama

In [6]:
converter = DocumentConverter()
chunker = HybridChunker()

doc = converter.convert(r"../airbus_report_of_the_board_of_directors_2024.pdf").document

texts = [chunk.text for chunk in chunker.chunk(doc)]

2025-11-10 22:32:17,541 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-11-10 22:32:19,952 - INFO - Going to convert document batch...
2025-11-10 22:32:19,953 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 44ae89a68fc272bc7889292e9b5a1bad
2025-11-10 22:32:19,972 - INFO - Loading plugin 'docling_defaults'
2025-11-10 22:32:19,980 - INFO - Registered picture descriptions: ['vlm', 'api']
2025-11-10 22:32:19,999 - INFO - Loading plugin 'docling_defaults'
2025-11-10 22:32:20,010 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2025-11-10 22:32:20,155 - INFO - rapidocr cannot be used because onnxruntime is not installed.
2025-11-10 22:32:20,157 - INFO - easyocr cannot be used because it is not installed.
2025-11-10 22:32:20,824 - INFO - Accelerator device: 'cpu'
[INFO] 2025-11-10 22:32:20,859 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2025-11-10 22:32:20,868 [RapidOCR] download_file.py:68: Ini

In [9]:
import torch

# Check if GPU or MPS is available
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA GPU is enabled: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("MPS GPU is enabled.")
else:
    raise OSError(
        "No GPU or MPS device found. Please check your environment and ensure GPU or MPS support is configured."
    )

OSError: No GPU or MPS device found. Please check your environment and ensure GPU or MPS support is configured.

In [11]:
def embed(text_chunk: str, model_name='mxbai-embed-large'):
    return ollama.embed(
        model=model_name,
        input=text_chunk
    )['embeddings'][0]

In [ ]:
from pymilvus import Collection, connections, DataType, FieldSchema, CollectionSchema, utility
import os

MILVUS_HOST = os.getenv('MILVUS_HOST', "localhost")
MILVUS_PORT = os.getenv('MILVUS_PORT', '19530')
COLLECTION_NAME = os.getenv('COLLECTION_NAME', 'report_chunks')
EMBED_DIM = os.getenv('EMBED_DIM', 1024)   

test_embeddings = ollama.embed(
  model='nomic-embed-text:v1.5',
  input='Llamas are members of the camelid family',
)

EMBED_DIM = len(test_embeddings.embeddings[0])
print(EMBED_DIM)

model_name = 'nomic-embed-text:v1.5'

class MilvusManager:
    _instance = None
    _collection = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super(MilvusManager, cls).__new__(cls)
            cls._instance._connect()
            cls._instance._init_collection()
        return cls._instance

    def _connect(self):
        '''Establish a single connection to Milvus.'''
        connections.connect("default", host=MILVUS_HOST, port=MILVUS_PORT)
        print(connections.list_connections())

    def _init_collection(self):
        '''Initialize or load collection schema once.'''
        fields = [
            FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
            FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=EMBED_DIM),
            FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=2000)
        ]
        schema = CollectionSchema(fields, description="Report text chunks with embeddings")

        try:
            self._collection = Collection(COLLECTION_NAME)
        except Exception:
            self._collection = Collection(name=COLLECTION_NAME, schema=schema)

        # Create index if not already present
        if not self._collection.has_index():
            self._collection.create_index(
                field_name="embedding",
                index_params={"metric_type": "COSINE", "index_type": "IVF_FLAT", "params": {"nlist": 128}}
            )

    def get_collection(self) -> Collection:
        '''Return the active collection object.'''
        return self._collection
    
    def semantic_search(self, query: str, top_k: int = 5):
        '''
        
        '''
        vector = embed(query, model_name)

        collection = self.get_collection()
        collection.load()

        results = collection.search(
            data=[vector],
            anns_field="embedding",
            param={"metric_type": "COSINE", "params": {"nprobe": 10}},
            limit=top_k,
            output_fields=["text"]
        )

        return [hit.entity.get("text") for hit in results[0]]